# Imports

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import re
from datetime import date as date
from requests import Session
from requests.exceptions import ConnectionError, Timeout, TooManyRedirects
import matplotlib.pyplot as plt
import json
from dotenv import load_dotenv
import os


# Retrieving Data from CoinMarketCap API

In [ ]:
# CoinMarketCap API_KEY
load_dotenv()
API_KEY = os.getenv("API_KEY")

In [ ]:
crypto_ids_url = 'https://pro-api.coinmarketcap.com/v1/cryptocurrency/map'
parameters = {
    'limit': '50',
}
headers = {
    'Accepts': 'application/json',
    'Accept-Encoding': 'deflate, gzip',
    'X-CMC_PRO_API_KEY': API_KEY,
}

session = Session()
session.headers.update(headers)

try:
    response = session.get(crypto_ids_url, params=parameters)
    data = json.loads(response.text)
    print(json.dumps(data, indent=4))
    session.close()
except (ConnectionError, Timeout, TooManyRedirects) as e:
    print(e)
    session.close()

In [ ]:
url = 'https://pro-api.coinmarketcap.com/v1/cryptocurrency/quotes/latest'
parameters = {
    'id': '4642',
    'convert': 'USD'
}
headers = {
    'Accepts': 'application/json',
    'Accept-Encoding': 'deflate, gzip',
    'X-CMC_PRO_API_KEY': API_KEY
}

session = Session()
session.headers.update(headers)

try:
    response = session.get(url, params=parameters, headers=headers)
    data = json.loads(response.text)
    print(json.dumps(data, indent=4))
    session.close()
except (ConnectionError, Timeout, TooManyRedirects) as e:
    print(e)
    session.close()

# Retrieving Data from Yahoo Finance

In [ ]:
start_date = '2019-09-18'
end_date = date.today()
symbol = 'HBAR-USD'
yf_data = yf.download(symbol, start_date, end_date)

In [ ]:
print(yf_data)

In [ ]:
yf_data.head()

In [ ]:
yf_data.tail()

In [ ]:
yf_data.describe()

In [ ]:
yf_data.info()

In [ ]:
def unify_column_names(name: str) -> any:
    """Converts column names to lowercase, replaces spaces/hyphens with underscores, and handles MultiIndex."""
    if isinstance(name, tuple):  # Handle MultiIndex columns
        return tuple(re.sub(r'[-\s]+', '_', n).lower() if isinstance(n, str) else n for n in name)
    return re.sub(r'[-\s]+', '_', name).lower() if isinstance(name, str) else name


# Apply transformation to column names
yf_data.columns = yf_data.columns.map(unify_column_names)

In [ ]:
print(f"Unified columns names: \n{yf_data.columns}")

# Working with Returns

In [ ]:
# Add a column
yf_data['returns'] = yf_data['close'].pct_change()
yf_data

In [ ]:
# Calculate Log Returns
yf_data['log_returns'] = np.log(yf_data['close'] / yf_data['close'].shift(1))
yf_data

In [ ]:
# Drop NA values
yf_data.dropna(inplace=True)
yf_data

In [ ]:
# Cumulative Sum of Log Returns
yf_data['log_returns_cum_sum'] = yf_data['log_returns'].cumsum()
yf_data

In [ ]:
# Normalize Log Returns
yf_data['normal_returns'] = np.exp(yf_data['log_returns_cum_sum']) - 1
yf_data

In [ ]:
# Fill NA values
yf_data.fillna(0, inplace=True)
yf_data

# Structure Conditions and Iterations
 

In [ ]:
# Create a new copy of yf_data
yf_data_new = yf_data.copy()
print(yf_data_new)

In [ ]:
# Work with certain rows and columns using iloc
yf_data_new.iloc[1:3, 0:]

# Conditionals

In [ ]:
# Add conditional statement
yf_data_new.loc[yf_data['close']['hbar_usd'].shift(-1) > yf_data['close']['hbar_usd'], 'target'] = 1
yf_data_new.loc[yf_data['close']['hbar_usd'].shift(-1) <= yf_data['close']['hbar_usd'], 'target'] = -1
yf_data_new

# Iterations

In [ ]:
# Iterate over data frame
i = 0
for index, row in yf_data_new.iterrows():
    print(index, row['close'], row['target'])
    if i >= 4:
        break
    i += 1

# Value Extraction, Multiple Adjustments, Save and Load

In [ ]:
# Extract close prices
yf_data_extract = yf_data_new.copy()
close_prices = yf_data_extract['close'].values
close_prices

In [ ]:
# Convert to a python list
close_prices_list = close_prices.tolist()
print(close_prices_list)

In [ ]:
# Get just one item from data frame
price = yf_data_extract['close']['hbar_usd'].iloc[1:2].item()
price

In [ ]:
# Multiple Adjustments
yf_data_extract[['open', 'close', 'volume']] = yf_data_extract[['open', 'close', 'volume']] / yf_data_extract[
    ['open', 'close', 'volume']].max()
yf_data_extract.head()

# Create Series and DataFrame

In [ ]:
# Create a Series
list_1 = [1, 3, 8, 4, 3]
series_1 = pd.Series(list_1)
series_1

In [ ]:
# Create a DataFrame
df = pd.DataFrame(series_1, columns=['Series'])
df

# Save and Load DataFrame

In [ ]:
# Save DataFrame
# yf_data_extract.to_csv('test.csv')

In [ ]:
# Load DataFrame
# df_test = pd.read_csv('test.csv')
# df_test.head()

# Data Extraction

In [ ]:
hbar_yf_data = yf.download(symbol, start_date, end_date)

In [ ]:
hbar_yf_data.head()

In [ ]:
hbar_yf_data.columns = hbar_yf_data.columns.map(unify_column_names)

In [ ]:
hbar_yf_data.info()

# Feature Adjustments

In [ ]:
# Add moving averages
hbar_yf_data["ma_12"] = hbar_yf_data["close"].rolling(12).mean()
hbar_yf_data["ma_21"] = hbar_yf_data["close"].rolling(21).mean()
hbar_yf_data.loc[hbar_yf_data["ma_12"] > hbar_yf_data["ma_21"], "signal"] = 1
hbar_yf_data.loc[hbar_yf_data["ma_12"] <= hbar_yf_data["ma_21"], "signal"] = 0
hbar_yf_data["signal"] = hbar_yf_data["signal"].shift(1)

In [ ]:
hbar_yf_data.head(40)

In [ ]:
# Add returns
hbar_yf_data["log_returns_benchmark"] = np.log(hbar_yf_data["close"] / hbar_yf_data["close"].shift(1))
hbar_yf_data["log_returns_prod_benchmark"] = hbar_yf_data["log_returns_benchmark"].cumsum()
hbar_yf_data["prod_benchmark"] = np.exp(hbar_yf_data["log_returns_prod_benchmark"]) - 1

In [ ]:
hbar_yf_data.head(40)

In [ ]:
# Add strategy returns
hbar_yf_data["log_returns_strategy"] = np.log(hbar_yf_data["open"].shift(-1) / hbar_yf_data["open"]).squeeze() * hbar_yf_data["signal"]
hbar_yf_data["log_returns_prod_strategy"] = hbar_yf_data["log_returns_strategy"].cumsum()
hbar_yf_data["prod_strategy"] = np.exp(hbar_yf_data["log_returns_prod_strategy"]) - 1
hbar_yf_data.tail(20)

In [ ]:
# Remove NaN's
hbar_yf_data.dropna(inplace=True)

In [ ]:
# Review DataFrame
print(f"DataFrame Length: {len(hbar_yf_data)}")
hbar_yf_data

In [ ]:
# Display Graph & Compare
fig = plt.figure(figsize=(15, 8))
plt.plot(hbar_yf_data["prod_benchmark"], label='Benchmark (Buy and Hold)')
plt.plot(hbar_yf_data["prod_strategy"], label='Strategy (12-week MA Cross Above 21-week MA)')
plt.xlabel('Date')
plt.ylabel('Percentage Returns')
plt.legend()
plt.show()

# Metrics

print(f"Benchmark Returns: {hbar_yf_data['prod_benchmark'].iloc[-1]}")
print(f"Strategy Returns: {hbar_yf_data['prod_strategy'].iloc[-1]}")

In [ ]:
# Calculate Sharpe Ratio
def calculate_sharp(returns: float) -> float:
    N = 255
    rf = 0.01
    SQRTN = np.sqrt(N)
    mean = returns.mean() * N
    sigma = returns.std() * SQRTN
    sharpe = round((mean - rf) / sigma, 3)
    return sharpe

In [ ]:
# Show Sharpe
bench_sharpe = calculate_sharp(hbar_yf_data["log_returns_benchmark"].values)
strategy_sharpe = calculate_sharp(hbar_yf_data["log_returns_strategy"].values)
print(f"Benchmark Sharpe: {bench_sharpe}")
print(f"Strategy Sharpe: {strategy_sharpe}")